# Image Resizing and Cloudflare R2 Re-upload

This notebook processes images listed in `data/ood/samples.json`:
1. **Download** images from their current `image_url`.
2. **Resize** them to a respectable dimension (max 1024px) for LLM readability while reducing file size.
3. **Re-upload** them to Cloudflare R2, overwriting the existing ones at the same path.

### Setup
Ensure you have your Cloudflare R2 credentials ready.

In [ ]:
# Install required libraries
%pip install boto3 Pillow requests tqdm

In [ ]:
import json
import os
import requests
import boto3
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from urllib.parse import urlparse

# R2 Configuration - Please fill in your secrets
R2_BUCKET = "thesis-bucket"
R2_PUBLIC_DOMAIN = "https://thesis-assets.andyathsid.com"
R2_SECRET_ACCESS_KEY = "YOUR_R2_SECRET_ACCESS_KEY"
R2_ACCESS_KEY_ID = "YOUR_R2_ACCESS_KEY_ID"
R2_ACCOUNT_ID = "YOUR_R2_ACCOUNT_ID"

# S3 client for Cloudflare R2
s3_client = boto3.client(
    service_name='s3',
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto" # R2 uses auto
)

# Load samples
samples_path = "data/ood/samples.json"
with open(samples_path, "r") as f:
    samples = json.load(f)

def resize_image(image_bytes, max_size=1024):
    img = Image.open(BytesIO(image_bytes))
    
    # Convert to RGB if necessary (e.g. for PNG/RGBA)
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")
        
    # Calculate new dimensions
    width, height = img.size
    if width > max_size or height > max_size:
        if width > height:
            new_width = max_size
            new_height = int(max_size * height / width)
        else:
            new_height = max_size
            new_width = int(max_size * width / height)
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    # Save to bytes
    output = BytesIO()
    img.save(output, format="JPEG", quality=85)
    return output.getvalue()

def process_images():
    for filename, info in tqdm(samples.items(), desc="Processing images"):
        image_url = info.get("image_url")
        if not image_url:
            continue
            
        try:
            # 1. Download
            response = requests.get(image_url)
            response.raise_for_status()
            
            # 2. Resize
            resized_data = resize_image(response.content)
            
            # 3. Upload back to R2
            # Extract key from URL or use the path in info
            # Based on image_url: https://thesis-assets.andyathsid.com/ood/evaluation/images/000_black_gram_anthracnose.jpg
            # The key should be: ood/evaluation/images/000_black_gram_anthracnose.jpg
            parsed_url = urlparse(image_url)
            r2_key = parsed_url.path.lstrip('/')
            
            s3_client.put_object(
                Bucket=R2_BUCKET,
                Key=r2_key,
                Body=resized_data,
                ContentType="image/jpeg"
            )
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Run the processing
# process_images()